# Algorithm 3 — Random Forest
**Member 3 | Telco Customer Churn Dataset**

---
### What is Random Forest?
Random Forest is an **ensemble** algorithm — it builds many Decision Trees (a 'forest') and combines their predictions to get a better, more reliable result.

Think of it like asking 100 different people the same question and going with the majority vote — the combined wisdom of many trees is more accurate than any single tree.

**How it works:**  
1. Creates many Decision Trees using random subsets of the training data  
2. Each tree gives a vote (Churn = Yes or No)  
3. The class with the most votes wins  

**Why use it here?**  
- Usually one of the most accurate algorithms  
- Handles overfitting better than a single Decision Tree  
- Provides excellent feature importance scores  
- Works well with tabular data like this dataset

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

print('All libraries imported successfully!')

---
## Step 2 — Load the Dataset

In [ ]:
df = pd.read_csv('Telco_csv.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

---
## Step 3 — Explore the Dataset

In [ ]:
print('Dataset Info:')
df.info()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Numeric feature distributions — compare churned vs non-churned customers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    df[df['Churn'] == 'No'][col].hist(ax=ax, alpha=0.6, color='steelblue',
                                       bins=30, label='No Churn')
    df[df['Churn'] == 'Yes'][col].hist(ax=ax, alpha=0.6, color='tomato',
                                        bins=30, label='Churn')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle('Numeric Features: Churn vs No Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_numeric_distributions.png', dpi=150)
plt.show()
print('Saved: rf_numeric_distributions.png')

In [ ]:
print('Churn value counts:')
print(df['Churn'].value_counts())

---
## Step 4 — Data Preprocessing

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
print('Shape after cleaning:', df.shape)

In [ ]:
df.drop('customerID', axis=1, inplace=True)

le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn'])

binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

df['gender'] = le.fit_transform(df['gender'])

multi_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print('Final dataset shape after encoding:', df.shape)

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')

# NOTE: Random Forest does NOT need feature scaling

---
## Step 5 — Train the Random Forest Model

In [ ]:
# n_estimators = number of trees in the forest
model_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

print('Model training complete!')
print(f'Number of trees : {model_rf.n_estimators}')

---
## Step 6 — Evaluate the Model

In [ ]:
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred_rf)
roc_auc  = roc_auc_score(y_test, y_prob_rf)

print('=' * 45)
print('         RANDOM FOREST RESULTS')
print('=' * 45)
print(f'  Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print('=' * 45)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model_rf, X_train, y_train, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Individual fold scores: {[round(s,4) for s in cv_scores]}')

---
## Step 7 — Visualizations

In [ ]:
# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Random Forest — Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=150)
plt.show()
print('Saved: rf_confusion_matrix.png')

In [ ]:
# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest — ROC Curve', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('rf_roc_curve.png', dpi=150)
plt.show()
print('Saved: rf_roc_curve.png')

In [ ]:
# 3. Top 15 Feature Importances
feat_imp = pd.Series(model_rf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
feat_imp.plot(kind='barh', color='darkorange', edgecolor='black')
plt.title('Random Forest — Top 15 Feature Importances\n(Higher = more influential in predicting churn)',
          fontsize=11, fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150)
plt.show()
print('Saved: rf_feature_importance.png')

In [ ]:
# 4. Number of Trees vs Accuracy
tree_counts = [10, 25, 50, 75, 100, 150, 200]
test_accs = []

for n in tree_counts:
    m = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(tree_counts, test_accs, 'o-', color='darkorange', linewidth=2)
plt.axvline(100, color='black', linestyle='--', alpha=0.5, label='Chosen n=100')
plt.xlabel('Number of Trees')
plt.ylabel('Test Accuracy')
plt.title('Random Forest — Number of Trees vs Accuracy', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('rf_ntrees_vs_accuracy.png', dpi=150)
plt.show()
print('Saved: rf_ntrees_vs_accuracy.png')

---
## Step 8 — Summary

In [ ]:
print('=' * 50)
print('         RANDOM FOREST — FINAL SUMMARY')
print('=' * 50)
print(f'  Algorithm         : Random Forest')
print(f'  Dataset           : Telco Customer Churn')
print(f'  Total Samples     : {len(df)}')
print(f'  Training Samples  : {len(X_train)}')
print(f'  Testing Samples   : {len(X_test)}')
print(f'  Features Used     : {X.shape[1]}')
print(f'  Number of Trees   : 100')
print(f'  Test Accuracy     : {accuracy*100:.2f}%')
print(f'  ROC-AUC Score     : {roc_auc:.4f}')
print(f'  CV Accuracy (5-fold): {cv_scores.mean()*100:.2f}%')
print('=' * 50)